# Layers: coordinates, coupling, measurements, and mathematics

A node can occur in several conditions, timepoints, or compartments. A
multilayer graph makes those placements explicit and can connect them. This
chapter develops the coordinate model, imports a two-layer network, attaches
measurements, selects regions, and computes with its supra-matrices.

Requires `annnet[polars,storage]` and the synthetic guide data. Run this notebook
independently. The examples use simplified signaling data and do not establish
biological relationships between experimental conditions.

## The coordinate model

Let $V$ be the set of node identities and $L_1,\ldots,L_d$ the labels of the
aspects. The possible layer coordinates form
$L=L_1\times\cdots\times L_d$. The placements actually present are a subset
$V_M\subseteq V\times L$. Declaring the labels does not create the whole product.

For `condition`, ERK in `('control',)` and ERK in `('treated',)` are different
placements of one node. With `time` and `condition`, a coordinate could be
`('6h', 'treated')`. Tuple order follows the aspect declarations; `G.at` lets
you address existing placements by aspect name instead.

| Edge endpoints | Role |
| --- | --- |
| Same layer | Intra-layer |
| Different nodes in different layers | Inter-layer |
| Same node in different layers | Coupling |

A present, isolated placement is different from an absent one. Use
`G.layers.place` to register isolated placements. Generic node attributes apply
to the identity across placements; `node_layers` attributes belong to one pair.

Slices name sets of existing elements. They suit multiple analysis results over
one network. Layers change endpoint identity, and are useful when a method
needs separate placements or inter-layer connections.

In [1]:
from pathlib import Path
import annnet as an
import polars as pl

# Repository root, notebook directory, or a downloaded notebook beside data/.
DATA = next((p for p in (
    Path('docs/guide/data'), Path('data')
) if (p / 'interactions.csv').is_file()), None)
if DATA is None:
    raise FileNotFoundError('Download the tutorial CSV files into a data/ directory.')


## Prepare contextual edge batches

Declare the condition labels, then convert table endpoints to explicit
`(node_id, layer_coordinate)` pairs. We reuse the same prior in both conditions
for this example. With condition-specific input, select the corresponding rows
instead. The ID suffix distinguishes each placed interaction.


In [2]:
interactions = pl.read_csv(DATA / 'interactions.csv')
conditions = ['control', 'treated']
M = an.Graph(directed=True, aspects={'condition': conditions})
M.attrs.backend = 'polars'

for condition in conditions:
    records = [
        {**row,
         'edge_id': f"{row['edge_id']}@{condition}",
         'source': (row['source'], (condition,)),
         'target': (row['target'], (condition,))}
        for row in interactions.iter_rows(named=True)
    ]
    an.add_edges_from_frame(M, records, edge_id='edge_id', sign='effect', slice='prior')

M.summary()


AnnNet summary: 10 node(s), 22 edge(s), 20 placement(s)
  directed default: True
  aspects (1): condition: [control, treated]
  layers occurring: 2
  slices (2): default (0n/0e), prior (10n/22e)
  attrs: edges: ['sign', 'confidence', 'evidence']
  edges (computed): binary=22; directed=22, undirected=0

There are 10 node identities, 20 placements, and 22 edges. ERK in control and
ERK in treatment share a node ID but occupy different coordinates. Explicit
endpoint pairs also avoid creating unintended placeholder placements.


In [3]:
assert (len(M.N), M.nv_supra, len(M.E)) == (10, 20, 22)
M.at('ERK', condition='treated')


('ERK', ('treated',))

## Attach a labeled measurement array

The CSV has nodes in rows and conditions in columns. `attach_values` expects
layers in rows and nodes in columns, so transpose the numerical block and pass
both sets of labels. The array is borrowed rather than copied.


In [4]:
measurements = pl.read_csv(DATA / 'node_measurements.csv')
measurements


node_id,control,treated
str,f64,f64
"""EGFR""",1.0,1.2
"""GRB2""",0.8,1.0
"""SOS1""",0.5,1.1
"""RAS""",0.4,1.2
"""RAF""",0.3,1.0
"""MEK""",0.2,0.9
"""ERK""",0.2,1.1
"""DUSP6""",0.1,0.7
"""AKT""",0.5,0.2


In [5]:
values = measurements.select(conditions).to_numpy().T
backing = M.layers.attach_values(
    {'response': values},
    layers=[(condition,) for condition in conditions],
    nodes=measurements['node_id'].to_list(),
)
block = M.layers.matrix('response', nodes=['ERK', 'DUSP6'])
print('Rows:', block.layers)
print('Columns:', block.nodes)
print(block.values)


Rows: [('control',), ('treated',)]
Columns: ['ERK', 'DUSP6']
[[0.2 0.1]
 [1.1 0.7]]


The rectangle is `[[0.2, 0.1], [1.1, 0.7]]`: control first, treatment
second; ERK first, DUSP6 second. Keep these labels with an algorithm's array
input so its output can be assigned back correctly. Attaching measurements alone
does not establish structural placements; the imports already created ours.


In [6]:
import numpy as np
assert np.allclose(block.values, [[0.2, 0.1], [1.1, 0.7]])


## Add coupling only when the model needs it

For illustration, connect each node to its counterpart in the other condition
with weight 0.2. This could express an analysis assumption about continuity;
it is not inferred from the measurements and is not evidence of causality.
Keep these introduced edges in a separate slice.


In [7]:
M.slices.add('coupling')
M.slices.active = 'coupling'
count = M.layers.couple('condition', kind='categorical', weight=0.2)
print('Coupling edges added:', count)
assert count == 10


Coupling edges added: 10


## Choose a boundary

The closed treatment view contains its 11 intra-layer edges and excludes
coupling to control. An open boundary also includes touching coupling edges
and their full endpoints. It therefore reaches beyond the treatment layer.


In [8]:
window = M.layers.where(condition='treated')
closed = M.view(layers=window)
opened = M.view(layers=window, boundary='open')
print('Closed:', len(closed.E), 'edges')
print('Open:', len(opened.E), 'edges')
assert len(closed.E) == 11 and len(opened.E) == 21


Closed: 11 edges
Open: 21 edges


## Supra-adjacency and its blocks

A supra-matrix puts the placements on ordinary matrix axes. For binary edges,
$A_{ij}$ is the sum of weights from placement $i$ to $j$. Undirected edges
contribute in both directions. Parallel edges add; their individual identities
remain available in the graph even when the matrix combines their weights.

With coordinates grouped by condition, our graph has the block form

$$A=\begin{pmatrix}A_{control}&\omega I\\\omega I&A_{treated}\end{pmatrix},
\qquad \omega=0.2.$$

The cross-layer identity blocks arise because each coupling joins the same node
across conditions. Other inter-layer relationships would have a different
pattern. This example contains no hyperedges; projecting hyperedges to a
pairwise operator requires its own policy.

In [9]:
A = M.layers.supra_adjacency()
A_intra = M.layers.build_intra_block()
A_inter = M.layers.build_inter_block()
A_coupling = M.layers.build_coupling_block()
assert A.shape == (20, 20)
assert np.allclose(A.toarray(), (A_intra + A_inter + A_coupling).toarray())

nodes = sorted(M.N)
control_rows = [M.layers.nl_to_row(node, ('control',)) for node in nodes]
treated_rows = [M.layers.nl_to_row(node, ('treated',)) for node in nodes]
cross_block = A.toarray()[np.ix_(control_rows, treated_rows)]
assert np.allclose(cross_block, 0.2 * np.eye(len(nodes)))
print('Cross-layer block:')
print(cross_block)

Cross-layer block:
[[0.2 0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.2 0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.2 0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.2 0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.2 0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.2 0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.2 0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.2 0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.2 0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0.2]]


Use `G.layers.nl_to_row` and `row_to_nl` with the full supra index. Do not
assume its ordering matches a separately built `G.A`, a filtered matrix, or
alphabetical order of bare node IDs. Named matrices such as
`G.matrices.adjacency()` provide their own maps. Save labels with numerical
results and rebuild positional lookups after structural edits.

In [10]:
row = M.layers.nl_to_row('ERK', ('control',))
assert M.layers.row_to_nl(row) == ('ERK', ('control',))
print('ERK in control occupies supra row', row)

ERK in control occupies supra row 6


## Laplacian and a random walk

For this nonnegative adjacency, set $D_{ii}=\sum_j A_{ij}$.
The combinatorial Laplacian is $L=D-A$. Its rows sum to zero.
`kind='norm'` provides $I-D^{-1/2}AD^{-1/2}$; symmetric spectral interpretations
require an appropriate undirected model, which our directed prior is not.

The transition matrix is $P=D^{-1}A$ on rows with positive degree. Here every
placement has a coupling neighbor, so all rows sum to one. For a graph with an
isolated placement, the API leaves its row zero; choose a dangling-node policy
before claiming that probability is conserved. Negative weights likewise need
a different interpretation.

In [11]:
L = M.layers.supra_laplacian(kind='comb')
P = M.layers.transition_matrix()
assert np.allclose(L.toarray(), np.diag(np.asarray(A.sum(axis=1)).ravel()) - A.toarray())
assert np.allclose(np.asarray(L.sum(axis=1)).ravel(), 0)
assert np.allclose(np.asarray(P.sum(axis=1)).ravel(), 1)

p0 = np.zeros(A.shape[0])
p0[M.layers.nl_to_row('ERK', ('control',))] = 1.0
p1 = M.layers.random_walk_step(p0)
assert np.isclose(p1.sum(), 1.0)
[(M.layers.row_to_nl(i), float(p1[i])) for i in np.flatnonzero(p1)]

[(('DUSP6', ('control',)), 0.8333333333333334),
 (('ERK', ('treated',)), 0.16666666666666669)]

From ERK in control, one step reaches DUSP6 in control with probability $1/1.2$
and ERK in treatment with probability $0.2/1.2$. These probabilities follow
from the chosen weights. They do not use the response array or the effect sign.

For a vector of node features, `diffusion_step(x, tau=...)` evaluates
$x-\tau Lx$. That is a different operation from moving a row probability vector
with $pP$. Choose a step size and directed/undirected model suitable for the
intended dynamics rather than treating these operators as interchangeable.

In [12]:
x = np.zeros(A.shape[0])
x[row] = 1.0
next_x = M.layers.diffusion_step(x, tau=0.1)
assert np.allclose(next_x, x - 0.1 * (L @ x))

## Several aspects and an ordered window

An aspect is categorical by default. Declare an order for time or dose when
queries should compare positions. The order is ordinal, not a declaration of
elapsed time. The following small table has two timepoints in treatment; the
control labels exist but no control placements are created.

In [13]:
from annnet import Aspect

rows = [
    {'edge_id': f'feedback@{time}',
     'source': ('DUSP6', (time, 'treated')),
     'target': ('ERK', (time, 'treated')),
     'effect': -1}
    for time in ('0h', '6h')
]
T = an.from_edge_frame(
    rows, edge_id='edge_id', sign='effect',
    aspects={'time': Aspect(['0h', '6h', '24h'], ordered=True),
             'condition': ['control', 'treated']},
)
print('Nodes:', len(T.N), 'placements:', T.nv_supra)
print(T.layers.where(time__lte='6h', condition='treated').layers)
assert T.nv_supra == 4
assert T.layers.couple('time', within={'condition': 'treated'}, weight=0.1) == 2

Nodes: 2 placements: 4
(('0h', 'treated'), ('6h', 'treated'))


Ordinal coupling joins consecutive declared time labels where both placements
exist. `both_present=False` allows it to create missing placements; use that
only when the model calls for them. Comparisons requiring order are refused on
categorical aspects. Explicit `pairs=` and `within=` can restrict coupling to
the coordinates you intend.

The costs to watch are placement count, edge count, and the size of requested
matrices. Attaching a large assay array does not require placing every assay
entity in every sample. Place the network entities needed for topology and
contextual writes, and read other measurements through their array labels.

## Save the placed measurements

Native saving materializes attached values for existing placements. All pairs
in this small matrix have placements, so both the layer coordinates and the
measurements should survive the round trip. A larger external assay may include
unplaced pairs; retain that assay separately.


In [14]:
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix='annnet-layers-') as directory:
    path = Path(directory) / 'layered.annnet'
    M.write(path)
    restored = an.read(path)

assert restored.nv_supra == 20
assert np.allclose(
    restored.layers.matrix('response', nodes=['ERK', 'DUSP6']).values, block.values
)
restored.summary()


AnnNet summary: 10 node(s), 32 edge(s), 20 placement(s)
  directed default: True
  aspects (1): condition: [control, treated]
  layers occurring: 2
  slices (3): default (0n/0e), prior (10n/22e), coupling (10n/10e)
  attrs: edges: ['sign', 'confidence', 'evidence', 'edge_kind']; node_layers: ['response']
  edges (computed): binary=32; directed=22, undirected=10

## API and related topics

[Layers API](../reference/core/layers.md) · [Attributes API](../reference/core/attributes.md) · [Computation and storage](computation-and-storage.ipynb)

[Guide contents](index.md) · [Sample data](data/index.md)